# AG_PRAXIS NB06e — The Forest and the Sequence Model on the Same Windows

Two models in this project are scored on different things. The random forest reads one
record and predicts its class, so it is scored on 1,229,711 test records. The sequence
model reads fifty consecutive records and predicts one class for the window, so it is
scored on 49,159 test windows. Their macro-F1 figures have been read against each other
so far, but they were never computed over the same items, so nothing about them is a
paired comparison.

This notebook puts both on the windows. Each window's fifty records already have a forest
prediction, saved when the forest was scored, so the forest gets a window-level prediction
by taking the majority vote of those fifty. That gives two predictions for every one of
the 49,159 windows, against one label, which is what a paired test and a count of where
one model is right and the other wrong both need.

Nothing is trained here and nothing is tuned. The windows were cut earlier, both sets of
predictions were saved when their models were scored, and this reads them back. No
hypothesis is tested. What comes out is a set of counts and a per-class table.

The usual first cell: Drive, the repository, and the commit this ran at.

In [ ]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

Every input is resolved and checked here, before anything is read. There are five of
them: the saved test records, the saved test windows, the forest's predictions, the
sequence model's predictions, and the label encoding NB08 wrote. The encoding is searched
for under its directory rather than assumed at a fixed path, because the file is the thing
that says which integer means which class and reading the wrong copy would mislabel every
table below.

A missing input stops the run here. A notebook that gets halfway through and then fails on
a path has already spent the session.

In [ ]:
import json

import numpy as np
import pandas as pd

from src import inventory as inv

CFG = inv.load_config(REPO_ROOT)
SEED = CFG["seed"]
WINDOW = int(CFG["sequence"]["window"])
STRIDE = int(CFG["sequence"]["stride"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])

FAST = os.environ.get("FAST", "0") == "1"
OUT_DIR = ARTIFACTS / ("NB06e_fast" if FAST else "NB06e")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(f"{ARTIFACTS} does not exist, so nothing this notebook writes survives.")


def first_existing(candidates, what):
    """The first candidate that is there, or a stop with every path that was tried."""
    found = next((Path(p) for p in candidates if Path(p).exists()), None)
    if found is None:
        raise FileNotFoundError(f"{what} not found. Looked in: {[str(p) for p in candidates]}")
    return found


def find_under(roots, filename, what):
    """Search each root for a named file at any depth, and stop if it is nowhere."""
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        found = sorted(root.rglob(filename))
        if found:
            return found[0]
    raise FileNotFoundError(f"{what} not found. Searched under: {[str(r) for r in roots]}")


MANIFEST_PATH = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB04_manifest.json",
     ARTIFACTS / "NB04" / "NB04_manifest.json"], "the manifest recording how the windows were cut")
RECORDS_NPZ = first_existing(
    [REPO_ROOT / "data" / "processed" / "records_test.npz",
     ARTIFACTS / "NB04" / "records_test.npz"], "the saved test records")
SEQUENCES_NPZ = first_existing(
    [REPO_ROOT / "data" / "processed" / "sequences_test.npz",
     ARTIFACTS / "NB04" / "sequences_test.npz"], "the saved test windows")
FOREST_DIR = first_existing(
    [REPO_ROOT / "results" / "NB05" / "forest_19class",
     ARTIFACTS / "NB05" / "forest_19class"], "the forest's saved predictions")
SEQUENCE_DIR = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB06" / "sequence_cnn_lstm_19class",
     ARTIFACTS / "NB06" / "sequence_cnn_lstm_19class"], "the sequence model's saved predictions")
LABEL_MAP_PATH = find_under(
    [REPO_ROOT / "data" / "processed" / "NB08", ARTIFACTS / "NB08"],
    "label_map.json", "the label encoding NB08 wrote")

INPUTS = {
    "manifest": MANIFEST_PATH,
    "records": RECORDS_NPZ,
    "windows": SEQUENCES_NPZ,
    "forest y_pred": FOREST_DIR / "y_pred.npy",
    "forest y_true": FOREST_DIR / "y_true.npy",
    "forest config": FOREST_DIR / "config.json",
    "sequence y_pred": SEQUENCE_DIR / "y_pred.npy",
    "sequence y_true": SEQUENCE_DIR / "y_true.npy",
    "label map": LABEL_MAP_PATH,
}
ABSENT = {name: str(path) for name, path in INPUTS.items() if not path.exists()}
if ABSENT:
    raise FileNotFoundError(f"these inputs are not where they were looked for: {ABSENT}")

for name, path in INPUTS.items():
    print(f"{name:<18} {path}")
print()
print(f"writing to        {OUT_DIR}")
print("the copy that goes into the repository afterwards is "
      "data/processed/NB06e/forest_window_comparison.json")

Seeding, and what the session is running on. Nothing here is trained and nothing needs an
accelerator, but the versions still go into what this writes, because a run whose
environment is remembered rather than recorded cannot be read back later against the
machine that produced it. One seeded choice is made below, a random sample of windows used
to check the vote against the rule it claims to follow, so the seed matters.

In [ ]:
import random

import keras
import tensorflow as tf

random.seed(SEED)
np.random.seed(SEED)


def accelerator():
    """What this ran on, by name, so a wall time can be read against the hardware."""
    devices = tf.config.list_physical_devices("GPU")
    if not devices:
        return "cpu"
    named = []
    for device in devices:
        details = tf.config.experimental.get_device_details(device)
        named.append(str(details.get("device_name", device.name)))
    return ", ".join(named)


ENVIRONMENT = {
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "backend": keras.backend.backend(),
    "accelerator": accelerator(),
    "run_date": RUN_DATE,
}

CHECKS = []


def confirm(name, ok, detail):
    """Record a check and its outcome, and stop the run if it failed.

    Every check this notebook makes goes through here, so the file it writes carries the
    list of what was verified rather than a claim that verification happened.
    """
    CHECKS.append({"check": name, "outcome": "pass" if ok else "fail", "detail": str(detail)})
    if not ok:
        raise AssertionError(f"{name}: {detail}")
    return ok


print(f"seeded with : {SEED}")
print(f"environment : {ENVIRONMENT}")

The one combiner, written down here before anything is measured.

Once there are two predictions for every window it is tempting to try rules for choosing
between them until one scores well, and a rule chosen that way tells you about the test set
rather than about the models. So exactly one rule is fixed now, and its class set comes
from somewhere outside this notebook: the five classes named in `PREREGISTRATION.md`
Amendment 6 as the ones the published run failed to detect, which are Recon-VulScan,
Recon-OS_Scan, MQTT-DDoS-Publish_Flood, Spoofing and MQTT-Malformed_Data.

The rule: when the sequence model predicts one of those five, take the sequence model's
prediction. Otherwise take the forest's window vote. Nothing this notebook measures feeds
back into the choice, and no other rule is tried.

What comes out of it is an illustration of what the difference between the two models is
worth, not a tuned model, and no hypothesis rests on it.

In [ ]:
COMBINER_CLASSES = ["Recon-VulScan", "Recon-OS_Scan", "MQTT-DDoS-Publish_Flood",
                    "Spoofing", "MQTT-Malformed_Data"]
COMBINER_RULE = ("take the sequence model's prediction whenever it names one of the five "
                 "classes of PREREGISTRATION.md Amendment 6, and the forest's window vote "
                 "otherwise")
COMBINER_SOURCE = ("PREREGISTRATION.md Amendment 6, the five classes below F1 0.50 in the "
                   "published run; chosen before anything here was measured")

print(COMBINER_RULE)
print()
for name in COMBINER_CLASSES:
    print(f"  {name}")

The label encoding, and the two sets of saved predictions.

Both models wrote their predictions as integer codes. Those codes mean nothing on their own,
so the encoding NB08 wrote is read first and everything else is checked against it: the
class list stored inside each array file, and the class list the forest recorded in its own
config when it ran. If any of the three disagreed then the codes do not carry the same
names in every file and none of the tables below would mean what they say.

In [ ]:
LABEL_MAP = json.loads(LABEL_MAP_PATH.read_text())
LABELS = [str(name) for name in LABEL_MAP["labels"]]
N_CLASSES = len(LABELS)
CODE = {name: i for i, name in enumerate(LABELS)}

FOREST_CONFIG = json.loads((FOREST_DIR / "config.json").read_text())
FOREST_CLASSES = [str(name) for name in FOREST_CONFIG["observed"]["classes"]]

FOREST_TRUE = np.load(FOREST_DIR / "y_true.npy").astype("int64")
FOREST_PRED = np.load(FOREST_DIR / "y_pred.npy").astype("int64")
SEQUENCE_TRUE = np.load(SEQUENCE_DIR / "y_true.npy").astype("int64")
SEQUENCE_PRED = np.load(SEQUENCE_DIR / "y_pred.npy").astype("int64")

confirm("the label map holds 19 classes in a fixed order", N_CLASSES == 19,
        f"{N_CLASSES} classes")
confirm("the forest's own config lists the same classes in the same order",
        FOREST_CLASSES == LABELS,
        "identical" if FOREST_CLASSES == LABELS else f"{FOREST_CLASSES} against {LABELS}")
confirm("the forest wrote as many predictions as it wrote labels",
        FOREST_TRUE.shape == FOREST_PRED.shape, f"{FOREST_TRUE.shape} and {FOREST_PRED.shape}")
confirm("the sequence model wrote as many predictions as it wrote labels",
        SEQUENCE_TRUE.shape == SEQUENCE_PRED.shape,
        f"{SEQUENCE_TRUE.shape} and {SEQUENCE_PRED.shape}")
confirm("every code in both prediction files is one of the 19",
        int(FOREST_PRED.min()) >= 0 and int(FOREST_PRED.max()) < N_CLASSES
        and int(SEQUENCE_PRED.min()) >= 0 and int(SEQUENCE_PRED.max()) < N_CLASSES,
        f"forest {int(FOREST_PRED.min())}..{int(FOREST_PRED.max())}, "
        f"sequence {int(SEQUENCE_PRED.min())}..{int(SEQUENCE_PRED.max())}")

print(f"forest    : {len(FOREST_TRUE):,} record predictions")
print(f"sequence  : {len(SEQUENCE_TRUE):,} window predictions")
print(f"classes   : {N_CLASSES}, read from {LABEL_MAP_PATH.name}")

Now the two index arrays. Neither the record features nor the window features are needed
here, only the labels and the position each row and each window came from, so the feature
blocks are left in the files rather than pulled into memory.

Each record carries the file it came from and its row number inside that file. Each window
carries the same file code and the row its first record sits at. The two files were written
in one pass, so the class list and the file list in them should be the same lists, and that
is checked rather than assumed.

The two size checks are the ones that say this is the comparison it claims to be: the
forest's labels are the test records element for element, and the sequence model's labels
are the test windows element for element.

In [ ]:
with np.load(RECORDS_NPZ, allow_pickle=False) as npz:
    RECORD_CLASSES = [str(v) for v in npz["classes"]]
    RECORD_FILES = [str(v) for v in npz["files"]]
    RECORD_Y = npz["y"].astype("int64")
    RECORD_FILE = npz["source_file"].astype("int64")
    RECORD_ROW = npz["row"].astype("int64")

with np.load(SEQUENCES_NPZ, allow_pickle=False) as npz:
    WINDOW_CLASSES = [str(v) for v in npz["classes"]]
    WINDOW_FILES = [str(v) for v in npz["files"]]
    WINDOW_Y = npz["y"].astype("int64")
    WINDOW_FILE = npz["source_file"].astype("int64")
    WINDOW_START = npz["start_row"].astype("int64")
    FILE_WINDOW = int(npz["window"])
    FILE_STRIDE = int(npz["stride"])

assert len(WINDOW_Y) == 49159, "the test partition is not the 49,159 windows this compares on"
assert len(RECORD_Y) == 1229711, "the test records are not the 1,229,711 rows the forest was scored on"

confirm("both array files carry the same class list as the label map",
        RECORD_CLASSES == LABELS and WINDOW_CLASSES == LABELS,
        "identical in both" if RECORD_CLASSES == LABELS == WINDOW_CLASSES else "they differ")
confirm("both array files carry the same file list, so a file code means the same thing in each",
        RECORD_FILES == WINDOW_FILES,
        f"{len(RECORD_FILES)} names, identical" if RECORD_FILES == WINDOW_FILES else "they differ")
confirm("the windows were cut at the window and stride this run expects",
        (FILE_WINDOW, FILE_STRIDE) == (WINDOW, STRIDE),
        f"window {FILE_WINDOW}, stride {FILE_STRIDE}")
confirm("the forest's saved labels are the test records, element for element",
        FOREST_TRUE.shape == RECORD_Y.shape and bool(np.array_equal(FOREST_TRUE, RECORD_Y)),
        f"{len(FOREST_TRUE):,} against {len(RECORD_Y):,} records, equal under this encoding")
confirm("the sequence model's saved labels are the test windows, element for element",
        SEQUENCE_TRUE.shape == WINDOW_Y.shape and bool(np.array_equal(SEQUENCE_TRUE, WINDOW_Y)),
        f"{len(SEQUENCE_TRUE):,} against {len(WINDOW_Y):,} windows, equal under this encoding")

print(f"{len(RECORD_Y):,} test records over {len(set(RECORD_FILE.tolist())):,} files")
print(f"{len(WINDOW_Y):,} test windows of {WINDOW} records, cut at stride {STRIDE}")

Finding a window's fifty records. A window's records are the records from the same file
whose row number falls in the half-open range from the window's start row to fifty rows
later. Searching for them one window at a time would be 49,159 searches through 1.2 million
rows, and it is not needed: the records of a file sit together in the array in row order, so
one table of where each file starts turns the search into arithmetic.

The table is built by walking the files once and checking, per file, that its records really
are one unbroken run and that their row numbers really do count up one at a time. A file is
not assumed to start at row zero either, because the split can hand one partition the first
part of a file and another partition the rest, so each file's first row number is read from
the array and used as the offset.

In [ ]:
BASE = np.full(len(RECORD_FILES), -1, dtype="int64")
ROW0 = np.zeros(len(RECORD_FILES), dtype="int64")
LENGTH = np.zeros(len(RECORD_FILES), dtype="int64")

BLOCKS = []
for file_code in np.unique(RECORD_FILE):
    where = np.flatnonzero(RECORD_FILE == file_code)
    contiguous = int(where[-1] - where[0] + 1) == len(where)
    rows = RECORD_ROW[where]
    counted = bool(np.array_equal(rows, np.arange(int(rows[0]), int(rows[0]) + len(rows))))
    confirm(f"the records of {RECORD_FILES[int(file_code)]} are one unbroken run in row order",
            contiguous and counted,
            f"{len(where):,} records at positions {int(where[0]):,} to {int(where[-1]):,}, "
            f"rows {int(rows[0]):,} to {int(rows[-1]):,}")
    BASE[int(file_code)] = int(where[0])
    ROW0[int(file_code)] = int(rows[0])
    LENGTH[int(file_code)] = len(where)
    BLOCKS.append({"file": RECORD_FILES[int(file_code)], "records": int(len(where)),
                   "first_row": int(rows[0]), "last_row": int(rows[-1])})

WINDOW_FILE_CODES = np.unique(WINDOW_FILE)
UNSEEN = [RECORD_FILES[int(c)] for c in WINDOW_FILE_CODES if BASE[int(c)] < 0]
confirm("every file the windows came from has its records in this partition", not UNSEEN,
        "all present" if not UNSEEN else f"missing {UNSEEN}")

print(pd.DataFrame(BLOCKS).to_string(index=False))

With the table in place a window's fifty records are one addition: the file's starting
position, plus how far the window's first row sits past the file's first row, and then the
fifty consecutive positions from there.

Three things are checked on the result, and any of them failing stops the run. That the
fifty positions hold the fifty row numbers the window says it covers, which is what makes
them the right fifty. That all fifty come from the window's own file. And that all fifty
carry the window's label, which they should, because a window takes the label of the file it
was cut from and so do the records.

Windows are cut at stride 25 and are fifty long, so consecutive windows share half their
records and a record can serve two windows. That is a property of how the windows were cut
and is left alone here. Each window votes over its own fifty.

In [ ]:
OFFSET = WINDOW_START - ROW0[WINDOW_FILE]
confirm("no window starts before its file's first record in this partition",
        bool((OFFSET >= 0).all()), f"smallest offset {int(OFFSET.min()):,}")
confirm("no window runs past its file's last record in this partition",
        bool((OFFSET + WINDOW <= LENGTH[WINDOW_FILE]).all()),
        f"largest overrun {int((OFFSET + WINDOW - LENGTH[WINDOW_FILE]).max()):,}")

MEMBERS = (BASE[WINDOW_FILE] + OFFSET)[:, None] + np.arange(WINDOW, dtype="int64")

confirm("every window maps to exactly fifty records",
        MEMBERS.shape == (len(WINDOW_Y), WINDOW), f"{MEMBERS.shape}")
confirm("those fifty records hold the fifty row numbers the window covers",
        bool(np.array_equal(RECORD_ROW[MEMBERS],
                            WINDOW_START[:, None] + np.arange(WINDOW, dtype="int64"))),
        "every row number matches")
confirm("all fifty come from the window's own file",
        bool((RECORD_FILE[MEMBERS] == WINDOW_FILE[:, None]).all()),
        "no window borrows a record from another file")
confirm("the fifty records' labels all equal the window's label",
        bool((RECORD_Y[MEMBERS] == WINDOW_Y[:, None]).all()),
        "every record agrees with its window")

USED, TIMES_USED = np.unique(MEMBERS, return_counts=True)
SHARED = int((TIMES_USED > 1).sum())
print(f"{len(WINDOW_Y):,} windows over {len(USED):,} distinct records")
print(f"{SHARED:,} of those records serve more than one window, at most "
      f"{int(TIMES_USED.max())} windows each, which is what stride {STRIDE} on a "
      f"{WINDOW}-record window gives")
print(f"{len(RECORD_Y) - len(USED):,} test records are in no window at all, being the tail "
      "of a file too short to start another one")

The vote, and the tie rule.

A window's forest prediction is the class named most often among its fifty records. Ties
are broken by `np.bincount(...).argmax()`, which returns the first position holding the
maximum, so a tie goes to the lowest class index. That is an arbitrary rule and it is stated
rather than hidden: with the classes in the order the label map fixes, a tie between two
classes always goes to whichever of them is alphabetically first.

The counts are taken for all windows at once rather than one at a time, so the vote is
checked against the rule it claims to follow on a random sample of windows, computed the
literal way with `np.bincount` per window.

How many windows were decided by a tie is reported, along with the distribution of how
lopsided the votes were, because a vote of 50 to 0 and a vote of 26 to 24 are both a
majority and they are not the same evidence.

In [ ]:
WINDOW_INDEX = np.arange(len(WINDOW_Y), dtype="int64")
VOTES = np.bincount(
    (WINDOW_INDEX[:, None] * N_CLASSES + FOREST_PRED[MEMBERS]).ravel(),
    minlength=len(WINDOW_Y) * N_CLASSES,
).reshape(len(WINDOW_Y), N_CLASSES)
FOREST_WINDOW = VOTES.argmax(axis=1).astype("int64")

confirm("every window's votes sum to fifty",
        bool((VOTES.sum(axis=1) == WINDOW).all()), f"all sum to {WINDOW}")

SAMPLE = np.random.default_rng(SEED).choice(
    len(WINDOW_Y), size=int(min(500, len(WINDOW_Y))), replace=False)
LITERAL = np.asarray([np.bincount(FOREST_PRED[MEMBERS[i]], minlength=N_CLASSES).argmax()
                      for i in SAMPLE], dtype="int64")
confirm("the vote agrees with np.bincount(...).argmax() taken window by window",
        bool(np.array_equal(LITERAL, FOREST_WINDOW[SAMPLE])),
        f"checked on {len(SAMPLE)} windows drawn at seed {SEED}")

TOP = VOTES.max(axis=1)
TIED = (VOTES == TOP[:, None]).sum(axis=1) > 1
N_TIED = int(TIED.sum())
PURITY = TOP / WINDOW

PURITY_BANDS = [(50, 50), (40, 49), (30, 39), (26, 29), (1, 25)]
PURITY_TABLE = pd.DataFrame([
    {"votes for the winner": f"{low}" if low == high else f"{low} to {high}",
     "windows": int(((TOP >= low) & (TOP <= high)).sum()),
     "share": float(((TOP >= low) & (TOP <= high)).mean())}
    for low, high in PURITY_BANDS
])
VOTE_HISTOGRAM = {int(v): int((TOP == v).sum()) for v in np.unique(TOP)}

print(f"windows decided by a tie : {N_TIED:,} of {len(WINDOW_Y):,} "
      f"({N_TIED / len(WINDOW_Y):.4%})")
print(f"unanimous windows        : {int((TOP == WINDOW).sum()):,}")
print(f"mean vote purity         : {float(PURITY.mean()):.4f}")
print(f"median vote purity       : {float(np.median(PURITY)):.4f}")
print(f"lowest vote purity       : {float(PURITY.min()):.4f}")
print()
print(PURITY_TABLE.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))

The first result. Per-class F1 for the forest voting at window level and for the sequence
model, side by side, over the same 49,159 windows with the same labels, and the test support
each class has in windows. Macro-F1 is the mean of the nineteen per-class figures, so a class
with four windows counts as much as a class with twelve thousand.

The forest's record-level macro-F1 is not this number and is not comparable to it. Voting
fifty records into one prediction changes both the unit and the class balance, so what is
below is the forest scored on windows and nothing else.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

ALL_LABELS = list(range(N_CLASSES))


def per_class_f1(y_true, y_pred):
    """Per-class F1 over all nineteen classes, and the support behind each."""
    _, _, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=ALL_LABELS, zero_division=0)
    return f1, support


F1_FOREST, SUPPORT = per_class_f1(WINDOW_Y, FOREST_WINDOW)
F1_SEQUENCE, _ = per_class_f1(WINDOW_Y, SEQUENCE_PRED)
MACRO_FOREST = float(F1_FOREST.mean())
MACRO_SEQUENCE = float(F1_SEQUENCE.mean())

SIDE_BY_SIDE = pd.DataFrame({
    "class": LABELS,
    "test windows": SUPPORT.astype("int64"),
    "F1 forest at window": F1_FOREST,
    "F1 sequence": F1_SEQUENCE,
})
SIDE_BY_SIDE["difference"] = SIDE_BY_SIDE["F1 forest at window"] - SIDE_BY_SIDE["F1 sequence"]

print(SIDE_BY_SIDE.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))
print()
print(f"macro-F1, forest at window : {MACRO_FOREST:.4f}")
print(f"macro-F1, sequence model   : {MACRO_SEQUENCE:.4f}")
print(f"difference                 : {MACRO_FOREST - MACRO_SEQUENCE:+.4f}")
print(f"accuracy, forest at window : {float((FOREST_WINDOW == WINDOW_Y).mean()):.4f}")
print(f"accuracy, sequence model   : {float((SEQUENCE_PRED == WINDOW_Y).mean()):.4f}")

The same windows split four ways. For every window there are two predictions and one label,
so each window falls into exactly one of four boxes: both models right, only the forest
right, only the sequence model right, or neither. Counting those per class says where the
two models differ, which a pair of F1 columns cannot: two models can score the same F1 on a
class and be right about different windows of it.

The two middle columns are the ones that matter for what follows. Where both are large the
models are making different mistakes on the same class.

In [ ]:
FOREST_OK = FOREST_WINDOW == WINDOW_Y
SEQUENCE_OK = SEQUENCE_PRED == WINDOW_Y

FOUR_WAY_ROWS = []
for c, name in enumerate(LABELS):
    here = WINDOW_Y == c
    FOUR_WAY_ROWS.append({
        "class": name,
        "test windows": int(here.sum()),
        "both correct": int((here & FOREST_OK & SEQUENCE_OK).sum()),
        "forest only": int((here & FOREST_OK & ~SEQUENCE_OK).sum()),
        "sequence only": int((here & ~FOREST_OK & SEQUENCE_OK).sum()),
        "neither": int((here & ~FOREST_OK & ~SEQUENCE_OK).sum()),
    })
FOUR_WAY = pd.DataFrame(FOUR_WAY_ROWS)
TOTALS = {
    "both correct": int((FOREST_OK & SEQUENCE_OK).sum()),
    "forest only": int((FOREST_OK & ~SEQUENCE_OK).sum()),
    "sequence only": int((~FOREST_OK & SEQUENCE_OK).sum()),
    "neither": int((~FOREST_OK & ~SEQUENCE_OK).sum()),
}

confirm("the four boxes account for every window once",
        sum(TOTALS.values()) == len(WINDOW_Y),
        f"{sum(TOTALS.values()):,} against {len(WINDOW_Y):,}")

print(FOUR_WAY.to_string(index=False))
print()
print("over all windows")
for name, count in TOTALS.items():
    print(f"  {name:<14} {count:>8,}   {count / len(WINDOW_Y):7.4%}")

One test on those counts. The windows where both models agree, right or wrong, say nothing
about which is better, so McNemar's test uses only the two discordant counts: the windows
only the forest gets right and the windows only the sequence model gets right. If the two
models were equally good those two counts would be two halves of the same coin.

This is one test at alpha 0.05. It is not part of a family and no multiple-comparison
correction applies to it. A McNemar helper already exists in `src/` from NB08, so it is
imported and reused rather than written again; which implementation ran, and whether a
continuity correction was applied inside it, both go into the file this writes.

In [ ]:
try:
    from src.significance import mcnemar as mcnemar_test

    MCNEMAR_IMPLEMENTATION = "src.significance.mcnemar, the helper NB08 uses"
except ImportError:
    from statsmodels.stats.contingency_tables import mcnemar as statsmodels_mcnemar

    def mcnemar_test(y_true, pred_a, pred_b, *, name_a="a", name_b="b"):
        """The same test through statsmodels, used only if the helper is not there."""
        correct_a, correct_b = pred_a == y_true, pred_b == y_true
        b = int(np.sum(correct_a & ~correct_b))
        c = int(np.sum(~correct_a & correct_b))
        result = statsmodels_mcnemar([[0, b], [c, 0]], exact=False, correction=True)
        return {"a": name_a, "b": name_b, "n_items": int(len(y_true)),
                "accuracy_a": float(correct_a.mean()), "accuracy_b": float(correct_b.mean()),
                "only_a_correct": b, "only_b_correct": c, "discordant": b + c,
                "statistic": float(result.statistic), "p_value": float(result.pvalue),
                "method": "chi-square with continuity correction",
                "favours": name_a if b > c else (name_b if c > b else "neither")}

    MCNEMAR_IMPLEMENTATION = "statsmodels.stats.contingency_tables.mcnemar"

ALPHA = 0.05
MCNEMAR = mcnemar_test(WINDOW_Y, FOREST_WINDOW, SEQUENCE_PRED,
                       name_a="forest at window", name_b="sequence model")
MCNEMAR_CORRECTION = ("Edwards continuity correction, applied inside the chi-square approximation"
                      if "continuity" in MCNEMAR["method"]
                      else "none; the exact binomial test was used and needs no correction")
SIGNIFICANT = bool(MCNEMAR["p_value"] < ALPHA)

confirm("the test was computed on the discordant windows and nothing else",
        MCNEMAR["discordant"] == TOTALS["forest only"] + TOTALS["sequence only"],
        f"{MCNEMAR['discordant']:,} discordant windows")

print(f"implementation   : {MCNEMAR_IMPLEMENTATION}")
print(f"method           : {MCNEMAR['method']}")
print(f"correction       : {MCNEMAR_CORRECTION}")
print(f"only forest right: {MCNEMAR['only_a_correct']:,}")
print(f"only sequence    : {MCNEMAR['only_b_correct']:,}")
print(f"discordant       : {MCNEMAR['discordant']:,}")
print(f"statistic        : {MCNEMAR['statistic']:.6g}")
print(f"p-value          : {MCNEMAR['p_value']:.6g}")
print(f"favours          : {MCNEMAR['favours']}")
print(f"at alpha {ALPHA}, one test, no correction family : "
      f"{'significant' if SIGNIFICANT else 'not significant'}")

The oracle union, which is a ceiling and not a system.

A window counts as correct here if either model got it right. No deployable system reaches
this, because reaching it would mean knowing which of the two models to believe on each
window, and knowing that is the same as knowing the answer. It is computed for one reason:
to put a number on how much the two models between them get right that neither gets right
alone.

To score it per class it needs an actual prediction per window, so it takes the prediction of
whichever model was correct, the forest's when both were, and the forest's when neither was.

In [ ]:
ORACLE = np.where(FOREST_OK, FOREST_WINDOW, np.where(SEQUENCE_OK, SEQUENCE_PRED, FOREST_WINDOW))
confirm("the oracle is correct exactly where at least one model was",
        bool(np.array_equal(ORACLE == WINDOW_Y, FOREST_OK | SEQUENCE_OK)),
        f"{int((ORACLE == WINDOW_Y).sum()):,} correct windows")

F1_ORACLE, _ = per_class_f1(WINDOW_Y, ORACLE)
MACRO_ORACLE = float(F1_ORACLE.mean())

ORACLE_TABLE = pd.DataFrame({
    "class": LABELS,
    "test windows": SUPPORT.astype("int64"),
    "F1 forest at window": F1_FOREST,
    "F1 sequence": F1_SEQUENCE,
    "F1 oracle union": F1_ORACLE,
})

print(ORACLE_TABLE.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))
print()
print(f"macro-F1, oracle union     : {MACRO_ORACLE:.4f}")
print(f"over the better of the two : {MACRO_ORACLE - max(MACRO_FOREST, MACRO_SEQUENCE):+.4f}")
print(f"accuracy, oracle union     : {float((ORACLE == WINDOW_Y).mean()):.4f}")
print()
print("This is a ceiling that no deployable system reaches. It is reported to size the "
      "difference between the two models and nothing more.")

The combiner, applied. The rule is the one fixed above and it is applied once. DoS-ICMP is
reported on its own line because it is the class the rule leaves entirely with the forest
while the sequence model's mistakes on the flooding classes are concentrated there, so it is
where a rule of this shape is most likely to cost something.

What this is: a post-hoc illustration of what the difference between the two models is
worth. What it is not: a tuned model, and not a result any hypothesis rests on.

In [ ]:
COMBINER_CODES = np.asarray([CODE[name] for name in COMBINER_CLASSES], dtype="int64")
USE_SEQUENCE = np.isin(SEQUENCE_PRED, COMBINER_CODES)
COMBINED = np.where(USE_SEQUENCE, SEQUENCE_PRED, FOREST_WINDOW)

confirm("the combiner took the sequence model's prediction exactly where the rule says",
        bool(np.array_equal(COMBINED[USE_SEQUENCE], SEQUENCE_PRED[USE_SEQUENCE]))
        and bool(np.array_equal(COMBINED[~USE_SEQUENCE], FOREST_WINDOW[~USE_SEQUENCE])),
        f"{int(USE_SEQUENCE.sum()):,} windows routed to the sequence model, "
        f"{int((~USE_SEQUENCE).sum()):,} to the forest")

F1_COMBINED, _ = per_class_f1(WINDOW_Y, COMBINED)
MACRO_COMBINED = float(F1_COMBINED.mean())

COMBINED_TABLE = pd.DataFrame({
    "class": LABELS,
    "test windows": SUPPORT.astype("int64"),
    "F1 forest at window": F1_FOREST,
    "F1 sequence": F1_SEQUENCE,
    "F1 combined": F1_COMBINED,
    "F1 oracle union": F1_ORACLE,
})

DOS_ICMP = CODE["DoS-ICMP"]
DOS_ICMP_WINDOWS = WINDOW_Y == DOS_ICMP
DOS_ICMP_ROUTED = int((DOS_ICMP_WINDOWS & USE_SEQUENCE).sum())
DOS_ICMP_FIGURES = {
    "test_windows": int(DOS_ICMP_WINDOWS.sum()),
    "f1_forest_at_window": float(F1_FOREST[DOS_ICMP]),
    "f1_sequence": float(F1_SEQUENCE[DOS_ICMP]),
    "f1_combined": float(F1_COMBINED[DOS_ICMP]),
    "f1_oracle_union": float(F1_ORACLE[DOS_ICMP]),
    "windows_routed_to_the_sequence_model": DOS_ICMP_ROUTED,
}

print(COMBINED_TABLE.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))
print()
print(f"macro-F1, forest at window : {MACRO_FOREST:.4f}")
print(f"macro-F1, sequence model   : {MACRO_SEQUENCE:.4f}")
print(f"macro-F1, combined         : {MACRO_COMBINED:.4f}")
print(f"macro-F1, oracle union     : {MACRO_ORACLE:.4f}")
print()
print(f"windows the rule sent to the sequence model : {int(USE_SEQUENCE.sum()):,} of "
      f"{len(WINDOW_Y):,}")
print()
print("DoS-ICMP under the combiner")
for name, value in DOS_ICMP_FIGURES.items():
    print(f"  {name:<38} {value if isinstance(value, int) else f'{value:.4f}'}")
print()
print("The combiner is a post-hoc illustration of what the difference between the two "
      "models is worth. It is not a tuned model and no hypothesis rests on it.")

Writing it down, so every figure above can be read again without running any of this.

In [ ]:
DOCUMENT = {
    "generated_by": "AG_PRAXIS_NB06e_forest_window_comparison.ipynb",
    "generated_on": RUN_DATE, "git_sha": GIT_SHA, "git_dirty": GIT_DIRTY,
    "seed": SEED, "is_fast_pass": bool(FAST), "environment": ENVIRONMENT,
    "trained": "nothing",
    "tests_no_hypothesis": True,
    "read": {name: str(path) for name, path in INPUTS.items()},
    "window": WINDOW, "stride": STRIDE,
    "classes": LABELS,
    "n_test_windows": int(len(WINDOW_Y)),
    "n_test_records": int(len(RECORD_Y)),
    "alignment": {
        "rule": "a window's fifty records are the records of the same source_file whose row "
                "falls in [start_row, start_row + 50)",
        "method": "one offset table per source file, then arithmetic indexing; no search",
        "blocks": BLOCKS,
        "distinct_records_used": int(len(USED)),
        "records_serving_more_than_one_window": SHARED,
        "most_windows_one_record_serves": int(TIMES_USED.max()),
        "records_in_no_window": int(len(RECORD_Y) - len(USED)),
    },
    "assertions": CHECKS,
    "vote": {
        "rule": "majority of the fifty per-record forest predictions",
        "tie_rule": "np.bincount(...).argmax(), which returns the first position holding the "
                    "maximum, so a tie goes to the lowest class index",
        "windows_decided_by_a_tie": N_TIED,
        "share_decided_by_a_tie": float(N_TIED / len(WINDOW_Y)),
        "unanimous_windows": int((TOP == WINDOW).sum()),
        "purity_mean": float(PURITY.mean()),
        "purity_median": float(np.median(PURITY)),
        "purity_min": float(PURITY.min()),
        "purity_bands": PURITY_TABLE.to_dict(orient="records"),
        "votes_for_the_winner_histogram": VOTE_HISTOGRAM,
    },
    "macro_f1": {
        "forest_at_window": MACRO_FOREST,
        "sequence": MACRO_SEQUENCE,
        "oracle_union": MACRO_ORACLE,
        "combined": MACRO_COMBINED,
    },
    "accuracy": {
        "forest_at_window": float(FOREST_OK.mean()),
        "sequence": float(SEQUENCE_OK.mean()),
        "oracle_union": float((ORACLE == WINDOW_Y).mean()),
        "combined": float((COMBINED == WINDOW_Y).mean()),
    },
    "per_class": [
        {"class": name, "test_windows": int(SUPPORT[c]),
         "f1_forest_at_window": float(F1_FOREST[c]), "f1_sequence": float(F1_SEQUENCE[c]),
         "f1_oracle_union": float(F1_ORACLE[c]), "f1_combined": float(F1_COMBINED[c])}
        for c, name in enumerate(LABELS)
    ],
    "four_way": FOUR_WAY.to_dict(orient="records"),
    "four_way_totals": TOTALS,
    "mcnemar": {
        "implementation": MCNEMAR_IMPLEMENTATION,
        "correction": MCNEMAR_CORRECTION,
        "alpha": ALPHA,
        "family": "none; one test, so no multiple-comparison correction applies",
        "significant": SIGNIFICANT,
        **MCNEMAR,
    },
    "oracle_union": {
        "definition": "a window counts correct if either model is correct; the prediction "
                      "taken is that of whichever model was right, and the forest's when "
                      "both or neither were",
        "status": "a ceiling that no deployable system reaches, reported to size the "
                  "difference between the two models and nothing more",
        "macro_f1": MACRO_ORACLE,
        "per_class_f1": {name: float(F1_ORACLE[c]) for c, name in enumerate(LABELS)},
    },
    "combiner": {
        "rule": COMBINER_RULE,
        "classes": COMBINER_CLASSES,
        "chosen_from": COMBINER_SOURCE,
        "status": "a post-hoc illustration of what the difference between the two models is "
                  "worth; not a tuned model, and no hypothesis rests on it",
        "fixed_before_any_result_cell_ran": True,
        "windows_routed_to_the_sequence_model": int(USE_SEQUENCE.sum()),
        "macro_f1": MACRO_COMBINED,
        "per_class_f1": {name: float(F1_COMBINED[c]) for c, name in enumerate(LABELS)},
        "dos_icmp": DOS_ICMP_FIGURES,
    },
}

ARTIFACT = OUT_DIR / "forest_window_comparison.json"
ARTIFACT.write_text(json.dumps(DOCUMENT, indent=1))
print(f"wrote {ARTIFACT}")
print(f"{len(CHECKS)} checks recorded, all passing, or this cell would not have been reached")

The ledger entry for this run.

In [ ]:
status = ("DO NOT ENTER, fast pass" if FAST else
          "reported result, working tree dirty" if GIT_DIRTY else
          "reported result, diagnostic; no model trained and no hypothesis tested")

ledger = f"""
### NB06e — the forest and the sequence model on the same windows ({RUN_DATE})

| field | value |
|---|---|
| notebook | AG_PRAXIS_NB06e_forest_window_comparison.ipynb |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SEED} |
| status | {status} |
| environment | {ENVIRONMENT} |
| trained | nothing; reads saved records, saved windows and both models' saved predictions |
| scored on | {len(WINDOW_Y):,} test windows, both models, same labels |
| forest at window | majority of its fifty per-record predictions, ties to the lowest class index |
| ties | {N_TIED:,} windows decided by a tie ({N_TIED / len(WINDOW_Y):.4%}); mean vote purity {float(PURITY.mean()):.4f} |
| macro-F1 forest at window | {MACRO_FOREST:.4f} |
| macro-F1 sequence | {MACRO_SEQUENCE:.4f} |
| both correct / forest only / sequence only / neither | {TOTALS['both correct']:,} / {TOTALS['forest only']:,} / {TOTALS['sequence only']:,} / {TOTALS['neither']:,} |
| McNemar | {MCNEMAR['method']} via {MCNEMAR_IMPLEMENTATION}; statistic {MCNEMAR['statistic']:.6g}, p {MCNEMAR['p_value']:.6g}, favours {MCNEMAR['favours']}; alpha {ALPHA}, one test, no correction family |
| oracle union | macro-F1 {MACRO_ORACLE:.4f}; a ceiling no deployable system reaches |
| combiner | macro-F1 {MACRO_COMBINED:.4f}; sequence model on the five Amendment 6 classes, forest otherwise; post-hoc illustration |
| DoS-ICMP under the combiner | F1 {DOS_ICMP_FIGURES['f1_combined']:.4f}, against {DOS_ICMP_FIGURES['f1_forest_at_window']:.4f} forest at window and {DOS_ICMP_FIGURES['f1_sequence']:.4f} sequence |
| artefact | {ARTIFACT} |
| hypothesis | none; diagnostic |
"""

print(ledger)